In [ ]:
import requests
import json
import time

auth = json.loads(mssparkutils.notebook.run("procore_auth"))
token = auth["token"]
COMPANY_ID = auth["company_id"]
headers = {
    "Authorization": f"Bearer {token}",
    "Procore-Company-Id": str(COMPANY_ID)
}

print("Auth successful" if token else "Auth failed")


StatementMeta(, 32dd475a-cb0c-49a5-b604-2e801657b35c, 3, Finished, Available, Finished, False)

Auth successful


In [2]:
projects_response = requests.get(
    "https://api.procore.com/rest/v1.0/projects",
    headers=headers,
    params={"company_id": COMPANY_ID}
)
projects = projects_response.json()
print(f"{len(projects)} projects found" if isinstance(projects, list) else "Failed to fetch projects")

StatementMeta(, 32dd475a-cb0c-49a5-b604-2e801657b35c, 4, Finished, Available, Finished, False)

16 projects found


In [3]:
all_commitments = []

for project in projects:
    project_id = project["id"]
    project_name = project["name"]
    print(f"Pulling commitments for: {project_name}")

    # Try Work Order Contracts (subcontracts)
    page = 1
    while True:
        response = requests.get(
            "https://api.procore.com/rest/v1.0/work_order_contracts",
            headers=headers,
            params={
                "project_id": project_id,
                "company_id": COMPANY_ID,
                "page": page,
                "per_page": 100
            }
        )

        rows = response.json()
        if not rows or isinstance(rows, dict):
            break

        for row in rows:
            row["project_id"] = project_id
            row["project_name"] = project_name
            row["commitment_type"] = "work_order"

        all_commitments.extend(rows)
        
        if len(rows) < 100:
            break
        page += 1
        time.sleep(0.3)

    # Try Purchase Order Contracts
    page = 1
    while True:
        response = requests.get(
            "https://api.procore.com/rest/v1.0/purchase_order_contracts",
            headers=headers,
            params={
                "project_id": project_id,
                "company_id": COMPANY_ID,
                "page": page,
                "per_page": 100
            }
        )

        rows = response.json()
        if not rows or isinstance(rows, dict):
            break

        for row in rows:
            row["project_id"] = project_id
            row["project_name"] = project_name
            row["commitment_type"] = "purchase_order"

        all_commitments.extend(rows)
        
        if len(rows) < 100:
            break
        page += 1
        time.sleep(0.3)

print(f"Done! Total commitments: {len(all_commitments)}")

StatementMeta(, 32dd475a-cb0c-49a5-b604-2e801657b35c, 5, Finished, Available, Finished, False)

Pulling commitments for: 1100 Fulton Street
Pulling commitments for: 11 ESSEX ST
Pulling commitments for: 337A & 337B West Broadway Rehabilitaion Work
Pulling commitments for: 360 Lexington 8th & 20th Floor
Pulling commitments for: 549 Munroe Av
Pulling commitments for: 64 MET OVAL PSC + 1410 MET STOREROOM
Pulling commitments for: Boys & Girls Club
Pulling commitments for: EMBANKMENT PHASE II
Pulling commitments for: Embankment + Revetment Apartments 270 & 310 10th Street NJ
Pulling commitments for: Lillipvt 45 Renwick St
Pulling commitments for: PCNA 711 11TH AVE
Pulling commitments for: Sandbox Test Project
Pulling commitments for: Standard Project Template
Pulling commitments for: SYMRISE - 15th & 16th Flr
Pulling commitments for: TEST - ABM SUBORDINATE
Pulling commitments for: VOCO HOTEL TSQ
Done! Total commitments: 264


In [4]:
import pandas as pd
import re

clean_rows = []
for row in all_commitments:
    clean_row = {}
    for key, value in row.items():
        if value is None:
            clean_row[key] = None
        elif isinstance(value, (dict, list)):
            clean_row[key] = json.dumps(value)
        elif isinstance(value, bool):
            clean_row[key] = str(value)
        elif isinstance(value, (int, float, str)):
            clean_row[key] = value
        else:
            clean_row[key] = str(value)
    clean_rows.append(clean_row)

def clean_column_name(col):
    col = col.strip()
    col = re.sub(r'[ ,;{}()\n\t=]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

pdf = pd.DataFrame(clean_rows)
pdf.columns = [clean_column_name(c) for c in pdf.columns]

for col in pdf.columns:
    if pdf[col].dtype == object:
        pdf[col] = pdf[col].astype(str).replace('None', None)

# Drop columns that are entirely null/empty
void_cols = [
    "approval_letter_date", "deleted_at", "execution_date",
    "letter_of_intent_date", "origin_code", "origin_data",
    "origin_id", "returned_date"
]
for col in void_cols:
    if col in pdf.columns:
        pdf = pdf.drop(columns=[col])

spark.sql("DROP TABLE IF EXISTS procore_commitments_raw")

df = spark.createDataFrame(pdf)
df.write.format("delta").mode("append").saveAsTable("procore_commitments_raw")

print("Saved to Bronze_Lakehouse successfully")

StatementMeta(, 32dd475a-cb0c-49a5-b604-2e801657b35c, 6, Finished, Available, Finished, False)

Saved to Bronze_Lakehouse successfully


In [5]:
display(spark.sql("DESCRIBE TABLE procore_commitments_raw"))

StatementMeta(, 32dd475a-cb0c-49a5-b604-2e801657b35c, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c8c7dd4-5cfe-4401-9efe-ca12fc021987)